# Prompt Management and Context Caching with Gemini


## Learning Objectives

1.  Learn how to use Agent Platform SDK to manage the lifecycle of prompt templates.
2.  Learn how to define, save, load and manage the prompts directly within Python code.
3.  Understand the concept of context caching and its benefits when working with large language models.
4.  Learn how to use the Agent Platform SDK to create and utilize cached content with Gemini models.
5.  Compare the performance of using cached content versus generating content from scratch, highlighting the speed and cost advantages.

## Overview
This notebook explores two key aspects of working with generative AI on Google Cloud. The first part focuses on Agent Platform's prompt management capabilities, explaining how to programmatically create, version, and organize prompt templates using the Agent Platform SDK. The second part introduces the Gemini API's context caching feature, designed to optimize requests with large, consistent initial contexts. 

## Basic Setup

In [ ]:
import datetime

from google import genai
from google.genai import types

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
MODEL = "gemini-3.5-flash"

## Context caching

The second section of this notebook demonstrates how to use context caching with Gemini models in Agent Platform. 

Context caching allows you to store the processed content, such as research papers, long videos or audios along with system instructions, so you don't have to re-process it every time. <br>
When you query the model, it can leverage the stored context, leading to faster response times and reduced resource consumption. This is particularly useful when working with large documents or when using the same context across multiple queries.

### Define the contents

Here we define the contents variable as a list of `Part` objects, each containing a reference to a research paper in PDF format stored in Google Cloud Storage.<br>
These are the papers that will be used for context caching.

In [ ]:
system_instruction = """
You are an expert researcher. You always stick to the facts in the sources provided, and never make up new facts.
Now look at these research papers, and answer the following questions.
"""

contents = [
    types.Part.from_uri(
        file_uri="gs://asl-public-data/data/generative-ai/pdf/2312.11805v3.pdf",
        mime_type="application/pdf",
    ),
    types.Part.from_uri(
        file_uri="gs://asl-public-data/data/generative-ai/pdf/2403.05530.pdf",
        mime_type="application/pdf",
    ),
]

### Create context caching

Let's create the cached content. It uses `client.caches.create` to set up a cache with specified parameters. The parameters are:

*   `model`: Specifies the Gemini model to use ("gemini-3.5-flash" in this case).
*   `config`: Basic configuration, which includes:
    *   `system_instruction`: Sets the instructions for how the model should behave.
    *   `contents`: The actual documents or other data you want to store in the cache.
    *   `ttl`: The time-to-live of the cache (60 minutes in this case), after which the cache will expire.
    *   `display_name`: A name for easy identification.

The output of this cell is the unique identifier `cached_content.name` that is used to retrieve cached content later.

In [ ]:
client = genai.Client(vertexai=True, location="global")

cached_content = client.caches.create(
    model=MODEL,
    config=types.CreateCachedContentConfig(
        system_instruction=system_instruction,
        contents=contents,
        ttl="3600s",
        display_name="example-cache",
    ),
)

print(cached_content.name)

Let's take a look at the created context cache!

In [ ]:
for cache in client.caches.list():
    print(cache)

### Generate without cached context

For comparison, let's first generate the answer **without** cached content and note the processing time.

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL, contents=contents + ["What are the papers about?"]
)

print(response.text)

### Generate with cached context

Now let's use the cached content to generate answers. The `cached_content` parameters refers to the created cached content. 

In [ ]:
%%time
response = client.models.generate_content(
    model=MODEL,
    contents="What are the papers about?",
    config=types.GenerateContentConfig(cached_content=cached_content.name),
)
print(response.text)

The output clearly demonstrates a substantial decrease in processing time. 

This performance gain amplifying as the volume of contextual information increases. By storing and reusing processed context, we achieve significant gains in efficiency, especially with larger contexts.

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.